# Robust phase estimation
This notebook demonstrates how to use Robust Phase Estimation (RPE) to estimate certain parameters of a standard single-qubit model.  RPE determines the phase of a target gate $U$ by iterated action of that gate on a superposition of eigenstates of $U$, and tolerates upwards of 30% error in counts, whatever its cause (statistical noise, calibration error).

## With the RPE protocol

For the $\pi/2$ phase of an $X$ rotation, the `smq1Q_Xpi2_rpe` model pack provides an experiment design, and the `RobustPhaseEstimation` protocol runs on its data like any other pyGSTi protocol. It needs only noisy $X_{\pi/2}$ gates and SPAM.

In [1]:
import pygsti
from pygsti.modelpacks import smq1Q_Xpi2_rpe, smq1Q_XY

exp_design = smq1Q_Xpi2_rpe.create_rpe_experiment_design(max_max_length=64)

# write an empty data object (creates a template to fill in)
pygsti.io.write_empty_protocol_data('../../tutorial_files/test_rpe_dir', exp_design, clobber_ok=True)

# fill in the template with simulated data (you would run the experiment and use actual data)
pygsti.io.fill_in_empty_dataset_with_fake_data(
    "../../tutorial_files/test_rpe_dir/data/dataset.txt",
    smq1Q_XY.target_model().depolarize(op_noise=0.01, spam_noise=0.1),
    num_samples=1000, seed=1234)

# read the data object back in, now with the experimental data
data = pygsti.io.read_data_from_dir('../../tutorial_files/test_rpe_dir/')

results = pygsti.protocols.rpe.RobustPhaseEstimation().run(data)
print(results.angle_estimate)

1.571377154153704


## With the lower-level functions

The functions in `pygsti.extras.rpe` expose the individual steps: circuit construction for a chosen RPE configuration, angle extraction from a model, and analysis of a data set. The rest of this notebook uses them to estimate the angles of an $X$/$Y$ gate set.

In [2]:
#Import relevant namespaces.

import pygsti
from pygsti.extras import rpe

import numpy as np

In [3]:
#Declare the particular RPE instance we are interested in
#(X and Y pi/2 rotations)
#(Prep and measurement are for the |0> state.   See below for prep and measure in |0> and |1>, respectively.)
rpeconfig_inst = rpe.rpeconfig_GxPi2_GyPi2_00

In [4]:
#Declare a variety of relevant parameters

target_model = pygsti.models.create_explicit_model_from_expressions([('Q0',)], ['Gx','Gy'],
                                  [ "X(pi/2,Q0)", "Y(pi/2,Q0)"])
target_model.set_all_parameterizations('full TP')
maxLengths_1024 = [1,2,4,8,16,32,64,128,256,512,1024]

stringListsRPE = rpe.rpeconstruction.create_rpe_angle_circuits_dict(10,rpeconfig_inst)

angleList = ['alpha','epsilon','theta']

numStrsD = {}
numStrsD['RPE'] = [6*i for i in np.arange(1,12)]

In [5]:
#Create noisy model
mdl_real = target_model.randomize_with_unitary(.01,seed=0)

In [6]:
#Extract noisy model angles
true_alpha = rpe.extract_alpha(mdl_real,rpeconfig_inst)
true_epsilon = rpe.extract_epsilon(mdl_real,rpeconfig_inst)
true_theta = rpe.extract_theta(mdl_real,rpeconfig_inst)

In [7]:
#Simulate dataset
N=1000
DS = pygsti.data.simulate_data(mdl_real,stringListsRPE['totalStrList'],N,sample_error='binomial',seed=1)

In [8]:
#Analyze dataset
resultsRPE = rpe.analyze_rpe_data(DS,mdl_real,stringListsRPE,rpeconfig_inst)

In [9]:
#Print results
print('alpha_true - pi/2 =',true_alpha-np.pi/2)
print('epsilon_true - pi/2 =',true_epsilon-np.pi/2)
print('theta_true =',true_theta)
print()
print('alpha_true - alpha_est_final =',resultsRPE['alphaErrorList'][-1])
print('epsilon_true - epsilon_est_final =',resultsRPE['epsilonErrorList'][-1])
print('theta_true - theta_est_final =',resultsRPE['thetaErrorList'][-1])

alpha_true - pi/2 = 0.005913261866022923
epsilon_true - pi/2 = 0.003042702315749546
theta_true = 0.010811833352494027

alpha_true - alpha_est_final = 4.7545269603599394e-05
epsilon_true - epsilon_est_final = 6.820432209853067e-05
theta_true - theta_est_final = 0.0020363381654932167


In [10]:
#Repeat above with prep and measure in |0> and |1>, respectively.)
rpeconfig_inst = rpe.rpeconfig_GxPi2_GyPi2_UpDn

In [11]:
#Declare a variety of relevant parameters
target_model = pygsti.models.create_explicit_model_from_expressions([('Q0',)], ['Gx','Gy'],
                                  [ "X(pi/2,Q0)", "Y(pi/2,Q0)"],
                                  effect_expressions=['1','0'])
target_model.set_all_parameterizations('full TP')
maxLengths_1024 = [1,2,4,8,16,32,64,128,256,512,1024]

stringListsRPE = rpe.rpeconstruction.create_rpe_angle_circuits_dict(10,rpeconfig_inst)

angleList = ['alpha','epsilon','theta']

numStrsD = {}
numStrsD['RPE'] = [6*i for i in np.arange(1,12)]

In [12]:
#Create noisy model
mdl_real = target_model.randomize_with_unitary(.01,seed=0)

In [13]:
#Extract noisy model angles
true_alpha = rpe.extract_alpha(mdl_real,rpeconfig_inst)
true_epsilon = rpe.extract_epsilon(mdl_real,rpeconfig_inst)
true_theta = rpe.extract_theta(mdl_real,rpeconfig_inst)

In [14]:
#Simulate dataset
N=1000
DS = pygsti.data.simulate_data(mdl_real,stringListsRPE['totalStrList'],N,sample_error='binomial',seed=1)

In [15]:
#Analyze dataset
resultsRPE = rpe.analyze_rpe_data(DS,mdl_real,stringListsRPE,rpeconfig_inst)

In [16]:
#Print results
print('alpha_true - pi/2 =',true_alpha-np.pi/2)
print('epsilon_true - pi/2 =',true_epsilon-np.pi/2)
print('theta_true =',true_theta)
print()
print('alpha_true - alpha_est_final =',resultsRPE['alphaErrorList'][-1])
print('epsilon_true - epsilon_est_final =',resultsRPE['epsilonErrorList'][-1])
print('theta_true - theta_est_final =',resultsRPE['thetaErrorList'][-1])

alpha_true - pi/2 = 0.005913261866022923
epsilon_true - pi/2 = 0.003042702315749546
theta_true = 0.010811833352494027

alpha_true - alpha_est_final = 4.7545269603599394e-05
epsilon_true - epsilon_est_final = 6.820432209853067e-05
theta_true - theta_est_final = 0.0020363381654932167
